# Feature Engineering and Normalization

This notebook is used for feature engineering and normalization of sequential data

In [1]:
import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Load Sequence Data

In [2]:
# Load data
data_dir = Path('../datasets')

with open(data_dir / 'sequences_obs_step_train.pkl', 'rb') as f:
    train_obs = pickle.load(f)
with open(data_dir / 'sequences_obs_step_val.pkl', 'rb') as f:
    val_obs = pickle.load(f)

with open(data_dir / 'sequences_daily_step_train.pkl', 'rb') as f:
    train_daily = pickle.load(f)
with open(data_dir / 'sequences_daily_step_val.pkl', 'rb') as f:
    val_daily = pickle.load(f)

print(f"ObsStep - Train: {len(train_obs)}, Val: {len(val_obs)}")
print(f"DailyStep - Train: {len(train_daily)}, Val: {len(val_daily)}")

ObsStep - Train: 1212, Val: 135
DailyStep - Train: 1212, Val: 135


## 2. Categorical Feature Encoding

Use `OneHotEncoder` to encode `crop_class`, `fertilization_class`, and `appl_class`.

In [3]:
# Collect unique values of categorical features
def collect_categorical_values(sequences, feature_name, is_static=True):
    values = []
    for seq in sequences:
        if is_static:
            values.append(seq['static_categorical'][feature_name])
        else:
            values.extend(seq['fertilization_categorical'][feature_name])
    return np.unique(values)

crop_class_values = collect_categorical_values(train_obs, 'crop_class', is_static=True)
fertilization_class_values = collect_categorical_values(train_obs, 'fertilization_class', is_static=False)
appl_class_values = collect_categorical_values(train_obs, 'appl_class', is_static=False)

print(f"crop_class: {crop_class_values}")
print(f"fertilization_class: {fertilization_class_values}")
print(f"appl_class: {appl_class_values}")

crop_class: ['barley' 'fruit' 'legume' 'maize' 'oat' 'oilplant' 'other_cereal' 'rice'
 'sugarbeet' 'vegetables' 'wheat']
fertilization_class: ['AN' 'CR' 'MA' 'NH4+' 'NO' 'NO3-' 'NPK' 'U' 'UAN']
appl_class: ['deep' 'no' 'surface']


In [4]:
# Create OneHotEncoders
encoders = {}
encoders['crop_class'] = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoders['crop_class'].fit(crop_class_values.reshape(-1, 1))
encoders['fertilization_class'] = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoders['fertilization_class'].fit(fertilization_class_values.reshape(-1, 1))
encoders['appl_class'] = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoders['appl_class'].fit(appl_class_values.reshape(-1, 1))

encoders['crop_class'].transform(crop_class_values.reshape(-1, 1))

array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])

## 3. Numerical Feature Transformation and Normalization

In [5]:
# Define conversion function
def symlog_transform(x):
    return np.sign(x) * np.log1p(np.abs(x))

def inverse_symlog_transform(y):
    return np.sign(y) * (np.exp(np.abs(y)) - 1)

def log1p_transform(x):
    return np.log1p(x)

# Feature Configuration
feature_config = {
    'transformations': {
        'Daily fluxes': 'symlog',
        'Prec': 'log1p',
        'Split N amount': 'log1p',
        'ferdur': 'log1p',
        'sowdur': 'log1p',
        'others': 'standard'
    }
}

In [6]:
# Collecting Numerical Features of the Training Set
def collect_features(sequences, key):
    return np.vstack([seq[key] if seq[key].ndim > 1 else seq[key].reshape(-1, 1) for seq in sequences])

# ObsStep Features
train_target_obs = collect_features(train_obs, 'target')
train_dynamic_obs = collect_features(train_obs, 'dynamic_numeric')
train_fert_obs = collect_features(train_obs, 'fertilization_numeric')
train_static_obs = np.vstack([seq['static_numeric'] for seq in train_obs])

def collect_masked_features(sequences, key):
    lst = []
    for seq in sequences:
        mask = seq['observed_mask']
        lst.append(seq[key][mask] if seq[key].ndim > 1 else seq[key].reshape(-1, 1)[mask])
    return np.vstack(lst)

# DailyStep Features
train_target_daily = collect_masked_features(train_daily, 'target')
train_dynamic_daily = collect_masked_features(train_daily, 'dynamic_numeric')
train_fert_daily = collect_masked_features(train_daily, 'fertilization_numeric')
train_static_daily = np.vstack([seq['static_numeric'] for seq in train_daily])

In [7]:
train_target_obs.shape, train_dynamic_obs.shape, train_fert_obs.shape, train_static_obs.shape

((31407, 1), (31407, 4), (31407, 4), (1212, 7))

In [8]:
train_target_daily.shape, train_dynamic_daily.shape, train_fert_daily.shape, train_static_daily.shape

((31191, 1), (31191, 4), (31191, 1), (1212, 7))

In [9]:
# Create Scalers
scalers = {}

# static features
scalers['static_numeric_obs'] = StandardScaler()
scalers['static_numeric_obs'].fit(train_static_obs)
scalers['static_numeric_daily'] = StandardScaler()
scalers['static_numeric_daily'].fit(train_static_daily)

# ObsStep dynamic features (Prec needs log1p)
train_dynamic_obs_t = train_dynamic_obs.copy()
train_dynamic_obs_t[:, 1] = log1p_transform(train_dynamic_obs[:, 1])
scalers['dynamic_numeric_obs'] = StandardScaler()
scalers['dynamic_numeric_obs'].fit(train_dynamic_obs_t)

# ObsStep fertilization features (Split N amount requires log1p)
train_fert_obs_t = train_fert_obs.copy()
train_fert_obs_t[:, :3] = log1p_transform(train_fert_obs[:, :3])
scalers['fertilization_numeric_obs'] = StandardScaler()
scalers['fertilization_numeric_obs'].fit(train_fert_obs_t)

# DailyStep dynamic features (Prec needs log1p)
train_dynamic_daily_t = train_dynamic_daily.copy()
train_dynamic_daily_t[:, 1] = log1p_transform(train_dynamic_daily[:, 1])
scalers['dynamic_numeric_daily'] = StandardScaler()
scalers['dynamic_numeric_daily'].fit(train_dynamic_daily_t)

# DailyStep fertilization features
train_fert_daily_t = log1p_transform(train_fert_daily)
scalers['fertilization_numeric_daily'] = StandardScaler()
scalers['fertilization_numeric_daily'].fit(train_fert_daily_t)

# target variable (symlog)
train_target_t = symlog_transform(train_target_obs)
scalers['target_obs'] = StandardScaler()
scalers['target_obs'].fit(train_target_t)

train_target_t = symlog_transform(train_target_daily)
scalers['target_daily'] = StandardScaler()
scalers['target_daily'].fit(train_target_t)

,copy,True
,with_mean,True
,with_std,True


In [10]:
seq = train_obs[0]
scalers['static_numeric_obs'].transform(seq['static_numeric'][None, :]).flatten()

array([ 0.60547357,  0.04674808,  0.32167732,  0.61441076, -0.36214276,
       -0.36557243,  0.21765746])

## 4. Apply to RNN dataset

In [ ]:
def transform_obs_sequences(seqs):
    result = []
    for seq in seqs:
        t_seq = {'seq_id': seq['seq_id'], 'seq_length': seq['seq_length']}
        t_seq['static_numeric'] = scalers['static_numeric_obs'].transform(seq['static_numeric'][None, :]).flatten().astype(np.float32)
        t_seq['static_categorical_encoded'] = encoders['crop_class'].transform([[seq['static_categorical']['crop_class']]]).flatten().astype(np.float32)

        dyn = seq['dynamic_numeric'].copy()
        dyn[:, 1] = log1p_transform(dyn[:, 1])
        t_seq['dynamic_numeric'] = scalers['dynamic_numeric_obs'].transform(dyn).astype(np.float32)

        fert = seq['fertilization_numeric'].copy()
        fert[:, :3] = log1p_transform(fert[:, :3])
        t_seq['fertilization_numeric'] = scalers['fertilization_numeric_obs'].transform(fert).astype(np.float32)

        t_seq['fertilization_categorical_encoded'] = {
            'fertilization_class': encoders['fertilization_class'].transform(seq['fertilization_categorical']['fertilization_class'].reshape(-1,1)).astype(np.float32),
            'appl_class': encoders['appl_class'].transform(seq['fertilization_categorical']['appl_class'].reshape(-1,1)).astype(np.float32)
        }

        tgt = symlog_transform(seq['target'])
        t_seq['target'] = scalers['target_obs'].transform(tgt.reshape(-1,1)).flatten().astype(np.float32)
        t_seq['target_original'] = seq['target']
        result.append(t_seq)
    return result

print("Transforming ObsStep...")
train_obs_t = transform_obs_sequences(train_obs)
val_obs_t = transform_obs_sequences(val_obs)
print(f"Done: {len(train_obs_t)} train, {len(val_obs_t)} val")

Transforming ObsStep...
Done: 1212 train, 135 val


In [12]:
train_obs_t[0]

{'seq_id': (np.int64(1), np.int64(1)),
 'seq_length': 16,
 'static_numeric': array([ 0.60547357,  0.04674808,  0.32167732,  0.61441076, -0.36214276,
        -0.36557243,  0.21765746]),
 'static_categorical_encoded': array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 'dynamic_numeric': array([[ 0.89670122, -0.64564988,  1.43042477, -0.82525373],
        [ 0.76410995, -0.64564988,  1.22109402,  0.18761993],
        [ 0.99982777, -0.64564988,  1.20713864, -0.82525373],
        [ 1.29447504, -0.64564988,  1.41646939,  0.18761993],
        [ 1.63331939, -0.64564988,  1.7374432 ,  0.13431079],
        [ 1.67751648, -0.0137804 ,  1.17922787,  0.02769251],
        [ 1.48599576, -0.64564988,  1.38855862, -1.4649634 ],
        [ 2.03109321,  0.04434175,  1.62580014, -0.02561663],
        [ 1.79537539, -0.64564988,  2.19797084,  0.40085649],
        [ 0.71991286, -0.64564988,  1.19318326,  0.40085649],
        [ 1.25027795, -0.64564988,  0.73265561, -0.66532631],
        [ 0.79357468, -0.64564

In [ ]:
def transform_daily_sequences(seqs):
    result = []
    for seq in seqs:
        t_seq = {
            'seq_id': seq['seq_id'], 'seq_length': seq['seq_length'],
            'min_day': seq['min_day'], 'max_day': seq['max_day'],
            'observed_mask': seq['observed_mask']
        }
        t_seq['static_numeric'] = scalers['static_numeric_daily'].transform(seq['static_numeric'][None, :]).flatten().astype(np.float32)
        t_seq['static_categorical_encoded'] = encoders['crop_class'].transform([[seq['static_categorical']['crop_class']]]).flatten().astype(np.float32)

        dyn = seq['dynamic_numeric'].copy()
        dyn[:, 1] = log1p_transform(dyn[:, 1])
        t_seq['dynamic_numeric'] = scalers['dynamic_numeric_daily'].transform(dyn).astype(np.float32)

        fert = log1p_transform(seq['fertilization_numeric'])
        t_seq['fertilization_numeric'] = scalers['fertilization_numeric_daily'].transform(fert).astype(np.float32)

        t_seq['fertilization_categorical_encoded'] = {
            'fertilization_class': encoders['fertilization_class'].transform(seq['fertilization_categorical']['fertilization_class'].reshape(-1,1)).astype(np.float32),
            'appl_class': encoders['appl_class'].transform(seq['fertilization_categorical']['appl_class'].reshape(-1,1)).astype(np.float32)
        }

        tgt = symlog_transform(seq['target'])
        t_seq['target'] = scalers['target_daily'].transform(tgt.reshape(-1,1)).flatten().astype(np.float32)
        t_seq['target_original'] = seq['target']
        result.append(t_seq)
    return result

print("Transforming DailyStep...")
train_daily_t = transform_daily_sequences(train_daily)
val_daily_t = transform_daily_sequences(val_daily)
print(f"Done: {len(train_daily_t)} train, {len(val_daily_t)} val")

Transforming DailyStep...
Done: 1212 train, 135 val


## 5. Save

In [14]:
processed_dir = data_dir / 'processed'
processed_dir.mkdir(exist_ok=True)

with open(processed_dir / 'sequences_obs_step_train_processed.pkl', 'wb') as f:
    pickle.dump(train_obs_t, f)
with open(processed_dir / 'sequences_obs_step_val_processed.pkl', 'wb') as f:
    pickle.dump(val_obs_t, f)
with open(processed_dir / 'sequences_daily_step_train_processed.pkl', 'wb') as f:
    pickle.dump(train_daily_t, f)
with open(processed_dir / 'sequences_daily_step_val_processed.pkl', 'wb') as f:
    pickle.dump(val_daily_t, f)

print(f"Saved to {processed_dir}")

Saved to ../datasets/processed


In [15]:
preprocessor_dir = Path('../preprocessor')
preprocessor_dir.mkdir(exist_ok=True)

with open(preprocessor_dir / 'encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)
with open(preprocessor_dir / 'scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)
with open(preprocessor_dir / 'feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)

print(f"\nPreprocessors saved to {preprocessor_dir.absolute()}")


Preprocessors saved to /Users/liaofeng/Documents/Codespace/N2O-pred/notebooks/../preprocessor
